# Non-local Networks — Non-local Neural Networks (2017)

_Papers · Computer Vision_

**Let every pixel directly attend to every other pixel in one shot — self-attention for images and video, with a residual wrapper.**

---

This notebook is a practice scaffold for the **Non-local Networks — Non-local Neural Networks (2017)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

We build the non-local block in three steps: (1) check the math on a tiny worked example, (2) write the embedded-Gaussian non-local block as a module, then (3) prove it solves a long-range copy task that a single convolution cannot.

### Step 1 — Check the math on a 3-position worked example

Before building anything, we reproduce the lesson's hand-computed example. With three positions whose features are just the identity values `[1, 2, 3]`, the affinity between positions `i` and `j` is `x_i * x_j`. Softmaxing those scores over `j` gives the **embedded-Gaussian** attention weights (Eqn. 3), and the response `y_i` is the weighted sum of the values (Eqn. 1). Row 1's weights should come out to `[0.09, 0.245, 0.665]` and `y_1 = 2.575`.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

# Sanity-check the lesson's worked example: N=3, identity embeddings, 1-dim features.
x = torch.tensor([1., 2., 3.])      # three positions, value = identity feature
scores = torch.outer(x, x)          # theta(x_i)*phi(x_j) = x_i * x_j  (identity embeds)
attn = torch.softmax(scores, dim=-1)  # f/C(x): embedded-Gaussian normalizer (Eqn. 3)
y = attn @ x                        # Eqn. 1: weighted sum of values g(x_j)=x_j

row1_weights = [round(w, 3) for w in attn[0].tolist()]
print("row-1 weights:", row1_weights)   # [0.09, 0.245, 0.665]
print("y_1 =", round(y[0].item(), 3))   # 2.575

### Step 2 — Build the embedded-Gaussian non-local block

Now we wrap that same computation into a reusable module that operates on `(B, C, N)` feature maps — `C` channels over `N` flattened positions. Three `1x1` convolutions produce the query (`theta`), key (`phi`), and value (`g`) embeddings; their all-pairs dot products become the attention scores. The output projection `Wz` is **zero-initialised** so the block starts as a pure identity (`z = x`) and only learns to mix positions as training pushes it to (Eqn. 6). A `use_nonlocal=False` switch lets us ablate the all-pairs path later.

In [ ]:
class NonLocalBlock1d(nn.Module):
    """Operates on (B, C, N): C channels over N positions (a flattened feature map)."""
    def __init__(self, channels, inner=None, use_nonlocal=True):
        super().__init__()
        inner = inner or max(1, channels // 2)
        self.use_nonlocal = use_nonlocal
        self.theta = nn.Conv1d(channels, inner, 1)   # query   (1x1 conv)
        self.phi   = nn.Conv1d(channels, inner, 1)   # key
        self.g     = nn.Conv1d(channels, inner, 1)   # value
        self.Wz    = nn.Conv1d(inner, channels, 1)   # output projection (Eqn. 6)
        # zero-init the projection -> block starts as the identity (z = x).
        nn.init.zeros_(self.Wz.weight)
        nn.init.zeros_(self.Wz.bias)

    def forward(self, x):
        if not self.use_nonlocal:
            return x                          # ABLATION: residual only, no all-pairs mixing
        B, C, N = x.shape
        theta = self.theta(x).permute(0, 2, 1)   # (B, N, inner)
        phi   = self.phi(x)                      # (B, inner, N)
        g     = self.g(x).permute(0, 2, 1)       # (B, N, inner)
        scores = torch.bmm(theta, phi)           # (B, N, N): theta_i . phi_j  for all i,j
        attn   = torch.softmax(scores, dim=-1)   # Eqn. 3: C(x)=sum_j f -> softmax over j
        y      = torch.bmm(attn, g).permute(0, 2, 1)   # Eqn. 1: weighted sum of values -> (B, inner, N)
        return self.Wz(y) + x                    # Eqn. 6: projection + residual

### Step 3 — Prove it on a long-range task a conv cannot solve

This toy task has 7 positions: channel 0 of the **far-right** cell holds a random value, and the **left end** must reproduce it — a dependency spanning 6 cells. A single radius-1 convolution sees only its immediate neighbours, so it physically cannot route that value in one layer. We train the network twice — once with the non-local block and once with it ablated to the identity — and compare the long-range MSE. The non-local version should drive the error near zero; the conv-only version stays stuck near the variance of the target.

In [ ]:
# A toy LONG-RANGE task a conv cannot solve in one layer.
# 7 positions; channel 0 carries a value, channel 1 is a one-hot "pointer" cell.
# Target at the LEFT end (pos 0) = the value sitting at the RIGHT end (pos 6): distance 6.
N, C = 7, 4

def make_batch(n=256):
    xb = torch.zeros(n, C, N)
    vals = torch.randn(n)        # the value to be transported
    xb[:, 0, N - 1] = vals       # value lives at the far-right position
    xb[:, 1, 0] = 1.0            # marker that pos 0 is the query position
    yb = vals                    # target the left end must reproduce
    return xb, yb

class Net(nn.Module):
    def __init__(self, use_nonlocal):
        super().__init__()
        self.local = nn.Conv1d(C, C, kernel_size=3, padding=1)   # radius-1: sees only +/-1 cell
        self.nl    = NonLocalBlock1d(C, use_nonlocal=use_nonlocal)
        self.read  = nn.Linear(C, 1)                             # read the left-end (pos 0) feature

    def forward(self, x):
        h = torch.relu(self.local(x))
        h = self.nl(h)           # one non-local block: global receptive field
        return self.read(h[:, :, 0]).squeeze(-1)   # predict from position 0 only

def train(use_nonlocal, steps=400):
    torch.manual_seed(0)
    net = Net(use_nonlocal)
    opt = torch.optim.Adam(net.parameters(), lr=1e-2)
    for _ in range(steps):
        xb, yb = make_batch()
        opt.zero_grad()
        loss = F.mse_loss(net(xb), yb)
        loss.backward()
        opt.step()
    with torch.no_grad():
        xb, yb = make_batch(1000)
        final = F.mse_loss(net(xb), yb).item()
    return final

mse_nl    = train(use_nonlocal=True)
mse_local = train(use_nonlocal=False)   # ABLATION: conv-only, no non-local block
print(f"with non-local block : long-range MSE {mse_nl:.4f}")
print(f"conv only (ablated)  : long-range MSE {mse_local:.4f}")
print("The non-local block routes the far-right value to the left end; the conv alone cannot.")
# with non-local block : long-range MSE ~0.00x   (solves it)
# conv only (ablated)  : long-range MSE ~1.0     (stuck near the variance of the target)
# (Exact numbers vary by hardware/seed; this is our small run, not the paper's reported result.)

## Visualize the data & results

_Can one non-local block capture a long-range dependency (left end depends on right end) that a single convolution physically cannot?_

### Step 1 — Rebuild the block, now recording its attention

This is the same experiment as above, made self-contained for the plot. The one addition is `self.last_attn`: the block stashes the softmax attention map on each forward pass so we can inspect *which* position the left-end query points at. Everything else — the 1x1 conv embeddings, the softmax affinity, and the zero-initialised residual projection — is identical to Step 2.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Reproduces the qualitative effect: one non-local block solves a long-range
# copy task a single radius-1 conv cannot, and its attention points at the source.
class NonLocalBlock1d(nn.Module):
    def __init__(self, c, inner=None, use_nonlocal=True):
        super().__init__()
        inner = inner or max(1, c // 2)
        self.on = use_nonlocal
        self.theta = nn.Conv1d(c, inner, 1)
        self.phi   = nn.Conv1d(c, inner, 1)
        self.g     = nn.Conv1d(c, inner, 1)
        self.Wz    = nn.Conv1d(inner, c, 1)
        nn.init.zeros_(self.Wz.weight)
        nn.init.zeros_(self.Wz.bias)
        self.last_attn = None          # stash the attention map for inspection

    def forward(self, x):
        if not self.on:
            return x
        B, C, N = x.shape
        th = self.theta(x).permute(0, 2, 1)
        ph = self.phi(x)
        gg = self.g(x).permute(0, 2, 1)
        attn = torch.softmax(torch.bmm(th, ph), dim=-1)   # Eqn. 3 normalizer
        self.last_attn = attn
        mixed = torch.bmm(attn, gg).permute(0, 2, 1)      # Eqn. 1: weighted sum of values
        return self.Wz(mixed) + x                         # Eqn. 6: projection + residual

### Step 2 — Train both variants and read off the attention

We re-create the 7-position copy task and the small network, train the non-local and conv-only variants, then print their long-range MSE. For the non-local run we also pull out `last_attn` for the left-end query: its weights over the 7 positions should spike at position 6, the source cell — direct evidence that the block learned to point at exactly the far-away value it needs.

In [ ]:
N, C = 7, 4

def make_batch(n=256):
    xb = torch.zeros(n, C, N)
    v = torch.randn(n)
    xb[:, 0, N - 1] = v       # value at the far-right position
    xb[:, 1, 0] = 1.0         # pointer marking the left-end query
    return xb, v

class Net(nn.Module):
    def __init__(self, on):
        super().__init__()
        self.local = nn.Conv1d(C, C, 3, padding=1)
        self.nl    = NonLocalBlock1d(C, use_nonlocal=on)
        self.read  = nn.Linear(C, 1)

    def forward(self, x):
        h = torch.relu(self.local(x))
        h = self.nl(h)
        return self.read(h[:, :, 0]).squeeze(-1)

def train(on, steps=400):
    torch.manual_seed(0)
    net = Net(on)
    opt = torch.optim.Adam(net.parameters(), lr=1e-2)
    for _ in range(steps):
        xb, yb = make_batch()
        opt.zero_grad()
        F.mse_loss(net(xb), yb).backward()
        opt.step()
    xb, yb = make_batch(1000)
    with torch.no_grad():
        mse = F.mse_loss(net(xb), yb).item()
    attn = net.nl.last_attn[0, 0].tolist() if on else None   # left query's weights over j
    return mse, attn

mse_nl, attn = train(True)
mse_loc, _   = train(False)
print("with non-local: MSE", round(mse_nl, 3))
print("conv only     : MSE", round(mse_loc, 3))
print("left-query attention over positions:", [round(a, 2) for a in attn])
# non-local solves the long-range copy and attends to position 6; conv-only stays near variance ~1.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The ablation. You have a setup where the output at the left end of a $1\times7$ feature row
            must reflect the value at the right end (six cells away), and a working non-local block solves it.
            Remove the non-local mixing &mdash; keep only the residual, so $z = x$ &mdash; and re-run. What do
            you expect, and what does the comparison demonstrate?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Disable only the non-local path: skip y = attn @ g and the $W_z$ projection, so the block returns just x (the residual). — _An honest ablation changes exactly one thing &mdash; the all-pairs attention &mdash; so any difference is attributable to it._
- Compare the left-end output (or the loss on the long-range target) with vs without the non-local mixing. — _Only the non-local path can route the right end's value to the left end in one layer; the local residual cannot._
- Note that a single $3\times3$ convolution sees only $\pm1$ cell, so without non-local mixing the far signal is unreachable in one layer. — _This is exactly the long-range-dependency gap the paper closes; the contrast isolates non-local attention as the cause._

**Answer:** Without the non-local mixing the block collapses to the identity ($z = x$), so the
                 left-end output cannot depend on the far-right value &mdash; the long-range target is
                 unfit. With it, the softmax affinity lets the left position attend directly to the right and
                 the target is matched. Since the two runs differ only by the non-local path, this isolates
                 all-pairs attention as the mechanism that captures long-range dependencies a
                 convolution cannot reach in one layer. The CODEVIZ panel shows this contrast.

</details>

**Problem 2.** Redo the worked example but for the response $y_3$ at position $3$, with the same inputs
            $x_1=1, x_2=2, x_3=3$, identity embeddings, embedded-Gaussian $f=\exp(x_i x_j)$. Which position
            dominates the weighted sum, and why?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Affinities $f(x_3,x_j)=\exp(3 x_j)$: $f_{31}=e^{3}\approx20.09$, $f_{32}=e^{6}\approx403.4$, $f_{33}=e^{9}\approx8103.1$. — _Larger $x_i x_j$ products give exponentially larger affinities; $x_3=3$ paired with $x_3=3$ gives the biggest exponent._
- Normalizer $C=20.09+403.4+8103.1=8526.6$; weights $w_{31}=0.0024$, $w_{32}=0.0473$, $w_{33}=0.9503$. — _Dividing by $C$ is the softmax; the exponential sharply favors the largest score._
- Weighted sum $y_3=0.0024\cdot1+0.0473\cdot2+0.9503\cdot3=0.0024+0.0946+2.851=2.948$. — _$y_3$ is pulled almost entirely toward position $3$'s own value because its self-affinity dominates._

**Answer:** Position $3$ dominates: weights $\approx[0.002, 0.047, 0.950]$ give
                 $y_3 \approx 2.95$. Because affinities are $\exp(x_i x_j)$, the largest product ($3\times3$)
                 produces an exponentially larger weight, so the softmax concentrates on position $3$. This is
                 the same softmax-sharpening behavior as scaled-dot-product attention &mdash; high-affinity
                 positions get nearly all the weight.

</details>

**Problem 3.** The paper says the embedded-Gaussian non-local operation is self-attention, yet also
            reports that a dot-product instantiation (Eqn. 4) &mdash; with $C(x)=N$ instead of the
            softmax denominator &mdash; works comparably. What does that tell you, and how does it relate to
            the cross-linked paper-attention lesson?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall that the embedded-Gaussian normalizer $C(x)=\sum_j f$ makes $f/C$ a softmax (the exact self-attention of paper-attention). — _Softmax weights are positive and sum to $1$ &mdash; a normalized, competitive attention._
- The dot-product variant uses $f=\theta(x_i)^\top\phi(x_j)$ and $C(x)=N$, a plain average of raw (un-exponentiated, un-competing) affinities. — _Dividing by the fixed count $N$ is not a softmax; weights need not sum to $1$ and do not compete via exponentiation._
- Both raise accuracy, so the gain comes mainly from the all-pairs, global mixing, not specifically from the softmax. — _The paper concludes "the attentional behavior (due to softmax) is not essential" &mdash; the long-range connectivity is what matters._

**Answer:** It tells you the long-range, all-pairs mixing is the essential ingredient, not the
                 softmax specifically: the embedded-Gaussian form recovers exactly the self-attention of the
                 paper-attention lesson, but a non-softmax dot-product normalization works about as well. So
                 non-local networks generalize self-attention &mdash; attention is one (important) instantiation
                 of a broader &lsquo;connect every position to every other&rsquo; operation.

</details>